In [ ]:
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. RUTAS DEFINITIVAS
# =========================================================
# Carpeta con todos los CSV que quieres convertir en tablas
CSV_FOLDER = r"C:\Users\juanb\Desktop\IronHack\SQL\Proyecto SQL\sql-database\tablas en csv"
# Base de datos destino
DB_PATH = r"C:\Users\juanb\Desktop\IronHack\SQL\Proyecto SQL\sql-database\spotify.db"
# Carpeta de salida para gráficos e informes
OUTPUT_DIR = r"C:\Users\juanb\Desktop\IronHack\SQL\Proyecto SQL\sql-database\output"

os.makedirs(OUTPUT_DIR, exist_ok=True)
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 7)

# =========================================================
# 2. IMPORTAR CADA CSV A SQLITE
# =========================================================
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print(f"Buscando archivos CSV en: {CSV_FOLDER}")
for filename in os.listdir(CSV_FOLDER):
    if filename.endswith(".csv"):
        file_path = os.path.join(CSV_FOLDER, filename)
        # Nombre de tabla basado en el nombre del archivo (sin .csv)
        table_name = os.path.splitext(filename)[0].replace(" ", "_")
        
        df_temp = pd.read_csv(file_path)
        df_temp.to_sql(table_name, conn, if_exists="replace", index=False)
        print(f"✅ Tabla '{table_name}' creada/actualizada desde {filename}")

# Obtener lista de tablas para validación
tablas_disponibles = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)['name'].tolist()
print(f"\nTablas disponibles para graficar: {tablas_disponibles}")

# =========================================================
# 3. VISUALIZACIÓN Y REPORTE (ANÁLISIS DE POPULARIDAD)
# =========================================================

# A) Top 15 Artistas (desde tabla albums)
if 'albums' in tablas_disponibles:
    query = "SELECT artists, COUNT(*) as total FROM albums GROUP BY artists ORDER BY total DESC LIMIT 15"
    df_albums = pd.read_sql(query, conn)
    plt.figure()
    sns.barplot(data=df_albums, x="total", y="artists", hue="artists", palette="magma", legend=False)
    plt.title("Top 15 Artistas (Volumen de canciones)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "01_viz_artistas.png"))
    plt.close()

# B) Top 15 Géneros por Popularidad Media (desde popularity_analysis)
if 'popularity_analysis' in tablas_disponibles:
    query = """
    SELECT track_genre, AVG(popularity) as avg_pop 
    FROM popularity_analysis 
    GROUP BY track_genre 
    ORDER BY avg_pop DESC 
    LIMIT 15
    """
    df_pop_genre = pd.read_sql(query, conn)
    plt.figure()
    sns.barplot(data=df_pop_genre, x="avg_pop", y="track_genre", hue="track_genre", palette="viridis", legend=False)
    plt.title("Top 15 Géneros más Populares (Media de Popularidad)")
    plt.xlabel("Popularidad Promedio")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "02_viz_popularidad_genero.png"))
    plt.close()

# C) Relación Popularidad vs Bailabilidad (desde popularity_analysis)
if 'popularity_analysis' in tablas_disponibles:
    df_scatter = pd.read_sql("SELECT danceability, popularity FROM popularity_analysis", conn)
    plt.figure()
    sns.scatterplot(data=df_scatter, x="danceability", y="popularity", alpha=0.4, color="#1DB954")
    plt.title("Relación: Bailabilidad vs Popularidad")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "03_viz_dance_vs_pop.png"))
    plt.close()

# =========================================================
# 4. COMPILAR INFORME FINAL
# =========================================================
# Obtener algún insight rápido para el reporte
insight_gen = ""
if 'popularity_analysis' in tablas_disponibles:
    top_gen = pd.read_sql("SELECT track_genre FROM popularity_analysis GROUP BY track_genre ORDER BY AVG(popularity) DESC LIMIT 1", conn)
    insight_gen = top_gen.iloc[0]['track_genre'] if not top_gen.empty else "N/A"

report = f"""# Informe de Análisis Spotify (Actualizado)
- **Total de tablas cargadas:** {len(tablas_disponibles)}
- **Tablas en DB:** {', '.join(tablas_disponibles)}

## Resumen del Proceso
- Se han importado los archivos CSV normalizados a SQLite.
- La tabla **popularity_analysis** ha sustituido el análisis previo, permitiendo cruzar datos de éxito comercial con métricas musicales.

## Insights Principales
- **Género Líder en Popularidad:** {insight_gen}
- Se generaron visualizaciones de volumen de artistas, ranking de géneros por éxito y correlaciones técnicas.
- Los archivos de imagen se encuentran en la carpeta `/output`.
"""

with open(os.path.join(OUTPUT_DIR, "reporte_final.md"), "w", encoding="utf-8") as f:
    f.write(report)

conn.close()
print(f"\n🎉 Proceso finalizado. El reporte y los gráficos están en: {OUTPUT_DIR}")

ModuleNotFoundError: No module named 'matplotlib'

In [3]:
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. RUTAS
# =========================================================
DB_PATH = r"C:\Users\juanb\Desktop\IronHack\SQL\Proyecto SQL\sql-database\spotify.db"
OUTPUT_DIR = r"C:\Users\juanb\Desktop\IronHack\SQL\Proyecto SQL\sql-database\output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# 2. CONFIGURACIÓN VISUAL
# =========================================================
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# =========================================================
# 3. CONEXIÓN
# =========================================================
conn = sqlite3.connect(DB_PATH)

# =========================================================
# 4. CONSULTAS ACTUALIZADAS
# =========================================================

# albums -> top artistas
q_albums = """
SELECT artists, COUNT(*) AS total_tracks
FROM albums
WHERE artists IS NOT NULL AND TRIM(artists) <> ''
GROUP BY artists
ORDER BY total_tracks DESC
LIMIT 15;
"""
df_albums = pd.read_sql_query(q_albums, conn)

# genre -> top géneros
q_genre = """
SELECT track_genre, COUNT(*) AS total_tracks
FROM genre
WHERE track_genre IS NOT NULL AND TRIM(track_genre) <> ''
GROUP BY track_genre
ORDER BY total_tracks DESC
LIMIT 15;
"""
df_genre = pd.read_sql_query(q_genre, conn)

# popularity_analysis -> Top canciones más populares
q_top_songs = """
SELECT track_name, popularity
FROM popularity_analysis
ORDER BY popularity DESC
LIMIT 15;
"""
df_top_songs = pd.read_sql_query(q_top_songs, conn)

# popularity_analysis -> Correlación Popularidad vs Bailabilidad (Danceability)
q_pop_dance = """
SELECT popularity, danceability
FROM popularity_analysis
WHERE popularity > 0;
"""
df_pop_dance = pd.read_sql_query(q_pop_dance, conn)

# Audio_Features -> promedio de métricas principales
q_features = """
SELECT
    AVG(danceability) AS danceability,
    AVG(energy) AS energy,
    AVG(acousticness) AS acousticness,
    AVG(valence) AS valence,
    AVG(liveness) AS liveness,
    AVG(speechiness) AS speechiness
FROM Audio_Features;
"""
df_features = pd.read_sql_query(q_features, conn)
df_features_long = df_features.melt(var_name="feature", value_name="avg_value")

# popularity_analysis -> popularidad media por género
q_pop_genre = """
SELECT
    track_genre,
    ROUND(AVG(popularity), 2) AS avg_popularity,
    COUNT(*) AS total_tracks
FROM popularity_analysis
GROUP BY track_genre
HAVING COUNT(*) >= 10
ORDER BY avg_popularity DESC
LIMIT 15;
"""
df_pop_genre = pd.read_sql_query(q_pop_genre, conn)

# =========================================================
# 5. GRÁFICOS
# =========================================================

# 1) Top Artistas
if not df_albums.empty:
    plt.figure()
    sns.barplot(data=df_albums, x="total_tracks", y="artists", hue="artists", palette="magma", legend=False)
    plt.title("Top 15 artistas con más canciones")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "01_albums_top_artistas.png"), dpi=300)
    plt.close()

# 2) Top Géneros
if not df_genre.empty:
    plt.figure()
    sns.barplot(data=df_genre, x="total_tracks", y="track_genre", hue="track_genre", palette="viridis", legend=False)
    plt.title("Top 15 géneros con más canciones")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "02_genre_top_generos.png"), dpi=300)
    plt.close()

# 3) Top Canciones por Popularidad (Desde popularity_analysis)
if not df_top_songs.empty:
    plt.figure()
    sns.barplot(data=df_top_songs, x="popularity", y="track_name", hue="track_name", palette="plasma", legend=False)
    plt.title("Top 15 canciones más populares")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "03_popularidad_top_canciones.png"), dpi=300)
    plt.close()

# 4) Dispersión: Popularidad vs Bailabilidad
if not df_pop_dance.empty:
    plt.figure()
    sns.scatterplot(data=df_pop_dance, x="danceability", y="popularity", alpha=0.5, color="#2ecc71")
    plt.title("Relación entre Bailabilidad y Popularidad")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "04_popularidad_vs_danceability.png"), dpi=300)
    plt.close()

# 5) Promedio de Audio Features
if not df_features_long.empty:
    plt.figure()
    sns.barplot(data=df_features_long, x="feature", y="avg_value", hue="feature", palette="Set2", legend=False)
    plt.title("Promedio de audio features globales")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "05_audio_features_promedio.png"), dpi=300)
    plt.close()

# 6) Popularidad Media por Género (Desde popularity_analysis)
if not df_pop_genre.empty:
    plt.figure()
    sns.barplot(data=df_pop_genre, x="avg_popularity", y="track_genre", hue="track_genre", palette="cubehelix", legend=False)
    plt.title("Top 15 géneros por popularidad media")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "06_popularidad_media_genero.png"), dpi=300)
    plt.close()

# =========================================================
# 6. INFORME
# =========================================================
report_lines = [
    "# Informe de análisis Spotify - Popularity Focus\n\n",
    "## Tablas utilizadas\n",
    "- albums\n",
    "- genre\n",
    "- Audio_Features\n",
    "- popularity_analysis (Tabla consolidada de métricas y éxito)\n\n",
    "## Visualizaciones generadas\n",
    "- Top artistas (Volumen de catálogo).\n",
    "- Top géneros (Diversidad de catálogo).\n",
    "- Top canciones por puntuación de popularidad.\n",
    "- Correlación entre métricas técnicas (danceability) y éxito comercial.\n",
    "- Promedio de audio features globales.\n",
    "- Ranking de géneros según su éxito promedio.\n\n",
    "## Insights clave\n"
]

if not df_albums.empty:
    report_lines.append(f"- El artista con mayor presencia en el dataset es **{df_albums.iloc[0]['artists']}**.\n")
if not df_top_songs.empty:
    report_lines.append(f"- La canción con mayor impacto actual es **{df_top_songs.iloc[0]['track_name']}** con {df_top_songs.iloc[0]['popularity']} puntos.\n")
if not df_pop_genre.empty:
    report_lines.append(f"- El género que domina el éxito comercial promedio es **{df_pop_genre.iloc[0]['track_genre']}**.\n")

with open(os.path.join(OUTPUT_DIR, "reporte_visualizaciones.md"), "w", encoding="utf-8") as f:
    f.writelines(report_lines)

conn.close()
print("✅ Gráficos e informe actualizados con 'popularity_analysis' en output/")

ModuleNotFoundError: No module named 'matplotlib'